# 08 — Manifold ve Bölmeli Tank Analizi

Manifold: check-valve kaçırması → bir tankta kayıp, eşleşen tankta kazanç.
Bölmeli: dolumda sac baskılanması → karşılıklı geçici hareket.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != 'eda':
    ROOT = ROOT / 'eda'
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_all
from utils.plots import setup_style

setup_style()
dfs = load_all()
tanks = dfs['tanks']
ue1t = dfs['ue1t_30min']

manifold = tanks[tanks['is_manifold']==1]
bolmeli = tanks[tanks['bolmeli']==1]
print('Manifoldlu tanklar:', len(manifold))
display(manifold[['istasyon_kodu','tank_no','manifold_grup_no','akaryakit_turu']])
print('\nBölmeli tanklar:', len(bolmeli))
display(bolmeli[['istasyon_kodu','tank_no','bolme_grup_no']])

In [ ]:
# Manifold çiftleri — aynı saatte ters yönlü kayıp/kazanç
pairs = []
for (ist, grp), g in manifold.groupby(['istasyon_kodu','manifold_grup_no']):
    tn = sorted(g['tank_no'].tolist())
    if len(tn) == 2:
        pairs.append((ist, tn[0], tn[1]))

print('Manifold çift sayısı:', len(pairs))
corrs = []
for ist, a, b in pairs[:6]:
    ua = ue1t[(ue1t.istasyon_kodu==ist)&(ue1t.tank_no==a)].set_index('saat_1')['kayip_kazanc']
    ub = ue1t[(ue1t.istasyon_kodu==ist)&(ue1t.tank_no==b)].set_index('saat_1')['kayip_kazanc']
    m = pd.concat([ua, ub], axis=1, keys=['a','b']).dropna()
    if len(m) > 50:
        corr = m['a'].corr(m['b'])
        corrs.append({'ist': ist, 'tank_a': a, 'tank_b': b, 'corr': corr})
        print(f'{ist} T{a}-T{b}: kayip_kazanc korelasyon = {corr:.3f}')

pd.DataFrame(corrs)

In [ ]:
# Örnek manifold çifti grafiği
if pairs:
    ist, a, b = pairs[0]
    ua = ue1t[(ue1t.istasyon_kodu==ist)&(ue1t.tank_no==a)].sort_values('saat_1')
    ub = ue1t[(ue1t.istasyon_kodu==ist)&(ue1t.tank_no==b)].sort_values('saat_1')
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(ua['saat_1'], ua['kayip_kazanc'].cumsum(), label=f'T{a}')
    ax.plot(ub['saat_1'], ub['kayip_kazanc'].cumsum(), label=f'T{b}')
    ax.set_ylabel('Kümülatif kayıp/kazanç')
    ax.legend()
    ax.set_title(f'{ist} Manifold çift T{a}-T{b}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Bölmeli çift IST_003 T2-T3
ba, bb = 2, 3
ist = 'IST_003'
ua = ue1t[(ue1t.istasyon_kodu==ist)&(ue1t.tank_no==ba)].sort_values('saat_1')
ub = ue1t[(ue1t.istasyon_kodu==ist)&(ue1t.tank_no==bb)].sort_values('saat_1')
m = ua.merge(ub, on='saat_1', suffixes=('_a','_b'))
m['toplam_kk'] = m['kayip_kazanc_a'] + m['kayip_kazanc_b']

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
m.plot(x='saat_1', y=['kayip_kazanc_a','kayip_kazanc_b'], ax=axes[0])
axes[0].set_ylabel('30dk kayıp/kazanç')
axes[0].set_title(f'{ist} Bölmeli T{ba}-T{bb}')
m.plot(x='saat_1', y='toplam_kk', ax=axes[1], color='purple')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('Toplam (a+b)')
plt.tight_layout()
plt.show()

Sonraki: anomali keşfi (`09_anomali_kesfi.ipynb`).